# Importing Libraries

In [15]:
import os
import pandas as pd
import numpy as np
import operator

from typing import Literal, TypedDict, Annotated
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv


In [16]:
# Load environment variables from .env file
load_dotenv()

True

In [17]:
# Load OpenAI API key
OPENAI_API_KEY = os.getenv("OPENAI_KEY")

# Creating State Graph

In [18]:
class AgentState(TypedDict):
    dataframe: pd.DataFrame
    numeric_description: str
    categorical_description: str
    combined_insights: str
    recommendation_report: str
    score: int
    feedback_report: str
    counter: int
    
    feedback_history: Annotated[list[str], operator.add]
    score_history: Annotated[list[int], operator.add]
    recommendation_history: Annotated[list[str], operator.add]


graph = StateGraph(AgentState)

# Pydantic class to collect feedback and score

In [19]:
class FeedbackScore(BaseModel):
    feedback: str = Field(description="Feedback to improve the recommendation report")
    score: int = Field(ge=0, le=10, description="Score between 0 and 10")

# Structured LLm Call

In [20]:
llm = ChatOpenAI(
    model="gpt-6-luna",
    api_key=OPENAI_API_KEY
)

structured_llm = llm.with_structured_output(FeedbackScore)

# Creating Functions

In [ ]:
def load_data(state: AgentState):
    file_path = os.path.join(os.getcwd(), "data.csv")
    dataframe = pd.read_csv(file_path)
    return {"dataframe": dataframe}

In [ ]:
def numerical_analysis(state: AgentState):
    dataframe = state["dataframe"]

    numeric_columns = dataframe.select_dtypes(include=np.number).columns
    numeric_data = dataframe[numeric_columns]

    analysis = pd.DataFrame({
        "data_type": numeric_data.dtypes.astype(str),
        "count": numeric_data.count(),
        "null_count": numeric_data.isnull().sum(),
        "null_percentage": (numeric_data.isnull().mean() * 100).round(2),
        "unique_values": numeric_data.nunique(),
        "mean": numeric_data.mean(),
        "median": numeric_data.median(),
        "mode": numeric_data.mode().iloc[0],
        "std_dev": numeric_data.std(),
        "min": numeric_data.min(),
        "25th_quartile": numeric_data.quantile(0.25),
        "75th_quartile": numeric_data.quantile(0.75),
        "max": numeric_data.max(),
        "iqr": numeric_data.quantile(0.75) - numeric_data.quantile(0.25),
        "skewness": numeric_data.skew()
    })

    numeric_description = analysis.to_string()

    return {"numeric_description": numeric_description}

# Creating nodes

In [26]:
graph.add_node("load_data", load_data)
graph.add_node("numeric_analysis", numeric_analysis)
graph.add_node("categorical_analysis", categorical_analysis)
graph.add_node("combined_analysis", combined_analysis)
graph.add_node("write_recommendation", write_recommendation)
graph.add_node("feedback_scoring", feedback_scoring)

NameError: name 'load_data' is not defined